# Analyze improvements after hyperparameter tuning
Let's check if each algorithm benefitted in any way from hyperparameter tuning.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re as regex
import seaborn as sns

from itertools import combinations
from scipy.stats import friedmanchisquare, wilcoxon
from sklearn.metrics import mean_squared_error, r2_score

# local script
from common import compute_corrected_ttest, holm_bonferroni

In [ ]:
# hard-coded values
table_filename = "improvements_hyperparameter_tuning.tex"
table_caption = "Comparison of performance between regressors, with base hyperparameters and after hyperparameter tuning."
table_label = "table:regressor-improvements-hyperparameter-tuning"

folder_default = "../results_server_default_hyperparameters_20260519"
folder_hyperparameters = "../results_server_hyperparameter_tuning_20260519"
#folder_hyperparameters = "../results_20260522_only_tree_models_5k_estimators"
file_results_folds = "openml_ctr23_statistics.csv"

output_folder = "../results_server_hyperparameter_tuning_20260519/latex_tables"
os.makedirs(output_folder, exist_ok=True)

alphas= [0.05, 0.01]

regressors_considered = None

# and a different highlight for each regressor
regressor_colors = {
    "PySRRegressor_default" : "ffd580",
    "PySRRegressor_validation" : "ffd580",
    "RandomForestRegressor" : "e6ffe6",
    #"XGBRegressor" : "7393b3",
    "XGBRegressor" : "aec2d6",
}

In [3]:
# let's start by reading the "openml_ctr23_statistics.csv" file in each folder
df_default = pd.read_csv(os.path.join(folder_default, file_results_folds), thousands=',')
df_hyperparameters = pd.read_csv(os.path.join(folder_hyperparameters, file_results_folds), thousands=',')

# then, we list all regressors appearing in each fold
regressors_default = sorted(df_default["regressor_name"].unique())
regressors_hyperparameters = sorted(df_hyperparameters["regressor_name"].unique())
regressors = [r for r in regressors_default if r in regressors_hyperparameters]

print("Regressors found in default file:", regressors_default)
print("Regressors found in hyperparameter tuning file:", regressors_hyperparameters)

# find the list of tasks which appear in both dataframes and are all completed (30 instances each)
completed_tasks = []
for t in df_default["task_id"].unique() :
    if len(df_default[df_default["task_id"] == t]) == len(regressors_default) * 10 and \
        len(df_hyperparameters[df_hyperparameters["task_id"] == t]) == len(regressors_hyperparameters) * 10 :
        completed_tasks.append(int(t))

# sort them by task_id, so that the table is coherent with the others
completed_tasks = sorted(completed_tasks)
print("Tasks which are completed in both files:", completed_tasks)

# let's start preparing the latex table
latex_table = r'\begin{table}[htb]' + "\n"
latex_table += r'\caption{' + table_caption + '}'
latex_table += r'\label{' + table_label + '}' + "\n"
latex_table += r'\centering' + "\n"
latex_table += r'\resizebox{0.99\textwidth}{!}{%' + "\n"
latex_table += r'\begin{tabular}{lr' + 'cc' * len(regressors) + '}' + "\n" # columns: task_id, dataset name, for each regressor (default/hyperparameters)
latex_table += r'\multirow{2}{*}{\textbf{Task ID}} & \multirow{2}{*}{\textbf{Dataset name}}'
for regressor in regressors :
    latex_table += r' & \multicolumn{2}{c}{\textbf{R2 ' + regressor.replace("_", "\_") + '}}'
latex_table += r'\\' + "\n"
latex_table += r' & '
for regressor in regressors :
    latex_table += r' & \textbf{Default} & \textbf{Enhanced}'
latex_table += r'\\ \hline \hline' + "\n"

# now, for each regressor and each task, we are interested in knowing if the regressor with
# tuned hyperparameters is performing statistically better than the regressor with default settings
for task in completed_tasks :
    # fill in the row of the table
    dataset_name = df_default[df_default["task_id"]==task]["dataset_name"].iloc[0]
    latex_table += str(int(task)) + r' & \texttt{' + dataset_name.replace("_", "\_") + '}'
    for regressor in regressors :
        print("Regressor \"%s\", on task %d" % (regressor, task))
        df_d = df_default[(df_default["task_id"] == task) & (df_default["regressor_name"] == regressor)]
        df_h = df_hyperparameters[(df_hyperparameters["task_id"] == task) & (df_hyperparameters["regressor_name"] == regressor)]

        mse_d = df_d["MSE"].values
        mse_h = df_h["MSE"].values
        mse_d_mean = np.mean(mse_d)
        mse_d_std = np.std(mse_d)
        mse_h_mean = np.mean(mse_h)
        mse_h_std = np.mean(mse_h)
        
        r2_d = df_d["R2"].values
        r2_h = df_h["R2"].values
        r2_d_mean = np.mean(r2_d)
        r2_d_std = np.std(r2_d)
        r2_h_mean = np.mean(r2_h)
        r2_h_std = np.std(r2_h)

        differences = mse_d - mse_h
        n_samples = int(df_d["n_samples"].iloc[0])
        n_train = n_samples * 0.9
        n_test = n_samples * 0.1
        degrees_of_freedom = 9
        t_stat, p_val = compute_corrected_ttest(differences, n_train, n_test, degrees_of_freedom=9)
        print("- p-value:", p_val)

        significant = {a : True if p_val < a else False for a in alphas}
        significant_list = [v for k, v in significant.items()]

        for a, s in significant.items() :
            if s :
                print("- The two performances **ARE** significantly different for alpha=%.2f" % a)
            else :
                print("- The two performances are **NOT** significantly different for alpha=%.2f" % a)

        latex_cell_color = r'\cellcolor[HTML]{' + regressor_colors[regressor] + r'}'
        
        cell_highlight = ""
        print("- r2_d_mean=%.4f, r2_h_mean=%.4f" % (r2_d_mean, r2_h_mean))
        print("- significant_list:", significant_list)
        if (True in significant_list) and (r2_d_mean > r2_h_mean) :
            cell_highlight = latex_cell_color
        latex_table += r' & ' + cell_highlight + '$%.4f \pm %.4f$' % (r2_d_mean, r2_d_std)
        
        cell_highlight = ""
        if (True in significant_list) and (r2_h_mean > r2_d_mean):
            cell_highlight = latex_cell_color
        latex_table += r' & ' + cell_highlight + '$%.4f \pm %.4f$' % (r2_h_mean, r2_h_std)

    # add the end of the latex row
    latex_table += r'\\' + "\n"

# wrap up the latex table and save it
latex_table += r'\hline' + "\n"
latex_table += r'\end{tabular}%' + "\n"
latex_table += r'}' + "\n"
latex_table += r'\end{table}' + "\n"

with open(os.path.join(output_folder, "table_comparison_base_vs_tuned.tex"), "w") as fp :
    fp.write(latex_table)


Regressors found in default file: ['PySRRegressor_default', 'PySRRegressor_validation', 'RandomForestRegressor', 'XGBRegressor']
Regressors found in hyperparameter tuning file: ['PySRRegressor_default', 'PySRRegressor_validation', 'RandomForestRegressor', 'XGBRegressor']
Tasks which are completed in both files: [361234, 361235, 361236, 361237, 361241, 361242, 361243, 361244, 361247, 361249, 361250, 361251, 361252, 361253, 361254, 361255, 361256, 361257, 361258, 361259, 361260, 361261, 361264, 361266, 361267, 361268, 361269, 361272, 361616, 361617, 361618, 361619, 361621, 361622, 361623]
Regressor "PySRRegressor_default", on task 361234
- p-value: 0.3414855283691748
- The two performances are **NOT** significantly different for alpha=0.05
- The two performances are **NOT** significantly different for alpha=0.01
- r2_d_mean=0.5074, r2_h_mean=0.4975
- significant_list: [False, False]
Regressor "PySRRegressor_validation", on task 361234
- p-value: 0.059490444116409885
- The two performance

c:\Research\symbolic-regression-vs-ensembles\src\common.py:122: RuntimeWarning: invalid value encountered in scalar divide
  t_stat = mean / std
